## 2026 EY AI & Data Challenge - TerraClimate Data Extraction Notebook

This notebooks demonstrates how to access the TerraClimate dataset. TerraClimate is a dataset of monthly climate and climatic water balance for global terrestrial surfaces from 1958 to the present. These data provide important inputs for ecological and hydrological studies at global scales that require high spatial resolution and time-varying data. All data have monthly temporal resolution and a ~4-km (1/24th degree) spatial resolution. This dataset is provided in Zarr format. 

For more information, visit: https://planetarycomputer.microsoft.com/dataset/terraclimate#overview 

In [1]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import os

<h2>Extracting TerraClimate Data Using API Calls</h2> <p align="justify"> The API-based method allows us to efficiently access <b>TerraClimate</b> data for specific regions and time periods through the <a href="https://planetarycomputer.microsoft.com/">Microsoft Planetary Computer</a>, ensuring scalability and reproducibility of the process. </p> <p align="justify"> Through the API, we can extract climate variables such as <b>Potential Evapotranspiration (PET)</b>, which represents the atmospheric demand for water. This variable provides critical insights into surface moisture balance and helps improve the accuracy of water quality modeling. </p> <p align="justify"> This approach ensures consistent, automated retrieval of high-resolution climate data that can be easily integrated with satellite-derived features for comprehensive environmental and hydrological analysis. </p>



<h3>Loading and Mapping TerraClimate Data:</h3>

<p>This section demonstrates how <b>TerraClimate climate variables</b>, such as <b>Potential Evapotranspiration (PET)</b>, are loaded and mapped to sampling locations:</p>

<ul>
  <li>The <b>load_terraclimate_dataset</b> function opens the TerraClimate Zarr/NetCDF dataset from the Microsoft Planetary Computer, handling storage options automatically.</li>
  <li>The <b>filterg</b> function filters the dataset for the desired time range (2011–2015) and spatial extent corresponding to the study region. The resulting data is converted to a pandas DataFrame with standardized column names.</li>
  <li>The <b>assign_nearest_climate</b> function maps each sampling location to its <b>nearest TerraClimate grid point</b> using a KD-tree and assigns the climate variable values corresponding to the closest time stamp.</li>
</ul>

<p>This workflow ensures efficient, reproducible retrieval of climate variables, while allowing participants to work with pre-extracted CSV files for faster benchmarking and analysis.</p>


In [2]:
def load_terraclimate_dataset():
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )

    return ds

In [3]:
# --- Filtering function (kept identical) ---
def filterg(ds, var):
    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))

    df_var_append = []
    for i in tqdm(range(len(ds_2011_2015.time))):
        df_var = ds_2011_2015.isel(time=i).to_dataframe().reset_index()
        df_var_filter = df_var[
            (df_var['lat'] > -35.18) & (df_var['lat'] < -21.72) &
            (df_var['lon'] > 14.97) & (df_var['lon'] < 32.79)
        ]
        df_var_append.append(df_var_filter)

    df_var_final = pd.concat(df_var_append, ignore_index=True)
    print(f"Filtering for {var} completed")

    df_var_final['time'] = df_var_final['time'].astype(str)

    # Column mapping
    col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
    df_var_final = df_var_final.rename(columns=col_mapping)

    return df_var_final


In [4]:
def filterg(ds, var):
    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))

    # Changed: make bbox robust to coordinate order
    lat_min, lat_max = -35.18, -21.72  # Changed
    lon_min, lon_max = 14.97, 32.79    # Changed

    lats = ds_2011_2015['lat'].values  # Changed
    lons = ds_2011_2015['lon'].values  # Changed

    lat_slice = slice(lat_min, lat_max) if lats[0] < lats[-1] else slice(lat_max, lat_min)  # Changed
    lon_slice = slice(lon_min, lon_max) if lons[0] < lons[-1] else slice(lon_max, lon_min)  # Changed

    ds_bbox = ds_2011_2015.sel(lat=lat_slice, lon=lon_slice)  # Changed

    df = ds_bbox.to_dataframe().reset_index()
    print(f"Filtering for {var} completed")

    df['time'] = df['time'].astype(str)
    col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
    df = df.rename(columns=col_mapping)

    return df

In [5]:
def assign_nearest_climate(sa_df, climate_df, var_name):
    if climate_df.empty:  # Changed
        raise ValueError(f"climate_df is empty for {var_name}. Check bbox/time slicing.") 
    
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)

    tree = cKDTree(climate_coords)
    _, idx = tree.query(sa_coords, k=1)

    nearest_points = climate_df.iloc[idx].reset_index(drop=True)

    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    # Changed: cache subsets by (lat, lon) once
    climate_groups = {k: g.sort_values('Sample Date') for k, g in climate_df.groupby(['Latitude', 'Longitude'])}  # Changed

    climate_values = []
    for i in tqdm(range(len(sa_df)), desc=f"Mapping {var_name.upper()} values"):
        key = (sa_df.loc[i, 'nearest_lat'], sa_df.loc[i, 'nearest_lon'])  # Changed
        subset = climate_groups.get(key)  # Changed
        if subset is None or subset.empty:
            climate_values.append(np.nan)
            continue

        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
        climate_values.append(subset.loc[nearest_idx, var_name])

    return pd.DataFrame({var_name: climate_values})

In [6]:
# --- Climate variable assignment function (unchanged logic) ---
def assign_nearest_climate(sa_df, climate_df, var_name):
    """
    Map nearest climate variable values to a new DataFrame 
    containing only the specified variable column.
    """
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)

    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)

    nearest_points = climate_df.iloc[idx].reset_index(drop=True)

    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    climate_values = []

    for i in tqdm(range(len(sa_df)), desc=f"Mapping {var_name.upper()} values"):
        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']

        subset = climate_df[
            (climate_df['Latitude'] == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]

        if subset.empty:
            climate_values.append(np.nan)
            continue

        nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
        climate_values.append(subset.loc[nearest_idx, var_name])

    output_df = pd.DataFrame({var_name: climate_values})

    
    return output_df

### Extracting features for the training dataset

In [11]:
Water_Quality_df=pd.read_csv('./data/water_quality_training_dataset.csv')
Water_Quality_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [6]:
Water_Quality_df.shape

(9319, 6)

In [7]:
# Load TerraClimate dataset, filter (time,region,parameter), filter for nearest parameter values
ds = load_terraclimate_dataset()

In [10]:
print(ds.head().compute())

<xarray.Dataset> Size: 9kB
Dimensions:  (time: 5, lat: 5, lon: 5, crs: 1)
Coordinates:
  * time     (time) datetime64[ns] 40B 1958-01-01 1958-02-01 ... 1958-05-01
  * lat      (lat) float64 40B 89.98 89.94 89.9 89.85 89.81
  * lon      (lon) float64 40B -180.0 -179.9 -179.9 -179.9 -179.8
  * crs      (crs) int16 2B 3
Data variables: (12/14)
    aet      (time, lat, lon) float32 500B nan nan nan nan ... nan nan nan nan
    def      (time, lat, lon) float32 500B nan nan nan nan ... nan nan nan nan
    pdsi     (time, lat, lon) float32 500B nan nan nan nan ... nan nan nan nan
    pet      (time, lat, lon) float32 500B nan nan nan nan ... nan nan nan nan
    ppt      (time, lat, lon) float64 1kB nan nan nan nan ... nan nan nan nan
    q        (time, lat, lon) float64 1kB nan nan nan nan ... nan nan nan nan
    ...       ...
    swe      (time, lat, lon) float64 1kB nan nan nan nan ... nan nan nan nan
    tmax     (time, lat, lon) float32 500B nan nan nan nan ... nan nan nan nan
    tmin  

In [25]:
tc_parameter = filterg(ds, 'pet')
print(tc_parameter.shape)
print(tc_parameter.head())

Filtering for pet completed
(8294640, 4)
  Sample Date   Latitude  Longitude         pet
0  2011-01-01 -21.729167  14.979167  169.400009
1  2011-01-01 -21.729167  15.020833  169.699997
2  2011-01-01 -21.729167  15.062500  170.000000
3  2011-01-01 -21.729167  15.104167  169.800003
4  2011-01-01 -21.729167  15.145833  169.199997


In [ ]:
import os

climate_vars = ['pet','ppt','aet','def','pdsi','q','soil','tmax','tmin','vpd']

Terraclimate_training_df = Water_Quality_df[['Latitude','Longitude','Sample Date']].copy()

for var in climate_vars:

    out_path = f"./source/tc_{var}_bbox_2011_2015.csv"
    if os.path.exists(out_path):
        continue

    tc_parameter = filterg(ds, var)
    tc_parameter.to_csv(out_path, index=False)
    del tc_parameter

Filtering for pet completed
Filtering for ppt completed
Filtering for aet completed
Filtering for def completed
Filtering for pdsi completed
Filtering for q completed
Filtering for soil completed
Filtering for tmax completed
Filtering for tmin completed
Filtering for vpd completed


In [ ]:
Terraclimate_training_df = Water_Quality_df[['Latitude','Longitude','Sample Date']].copy()

for var in climate_vars:
    tc_parameter = pd.read_csv(f"./source/tc_{var}_bbox_2011_2015.csv")  # Changed
    temp_df = assign_nearest_climate(Water_Quality_df, tc_parameter, var)  # Changed
    Terraclimate_training_df[var] = temp_df[var]  # Changed

Terraclimate_training_df.to_csv('./data/terraclimate_features_training.csv', index=False)  # Changed

Mapping VPD values: 100%|██████████| 9319/9319 [00:00<00:00, 11273.35it/s]


In [ ]:
Terraclimate_training_df['Latitude'] = Water_Quality_df['Latitude']
Terraclimate_training_df['Longitude'] = Water_Quality_df['Longitude']
Terraclimate_training_df['Sample Date'] = Water_Quality_df['Sample Date']
Terraclimate_training_df = Terraclimate_training_df[['Latitude', 'Longitude', 'Sample Date'] + climate_vars]
Terraclimate_training_df.to_csv('./data/terraclimate_features_training.csv', index=False)

In [34]:
# Preview File
Terraclimate_training_df.head()

,Latitude,Longitude,Sample Date,pet,ppt,aet,def,pdsi,q,soil,tmax,tmin,vpd
0,-28.760833,17.730278,02-01-2011,174.2,32.7,31.100000,143.100000,3.65,1.6,0.0,36.100000,22.689999,2.92
1,-26.861111,28.884722,03-01-2011,124.1,51.1,53.800000,70.300000,0.66,2.6,12.8,27.160000,13.219999,0.95
2,-26.450000,28.085833,03-01-2011,127.5,62.7,60.800000,66.700005,-1.16,3.1,6.8,27.519999,14.090000,1.02
3,-27.671111,27.236944,03-01-2011,129.7,84.2,83.200005,46.500000,2.84,4.2,7.2,28.869999,14.639999,1.22
4,-27.356667,27.286389,03-01-2011,129.2,78.0,77.300000,51.900000,2.65,3.9,7.8,28.670000,14.690000,1.18


In [30]:
print(Terraclimate_training_df.shape)
print(Terraclimate_training_df.columns.tolist())

(9319, 13)
['Latitude', 'Longitude', 'Sample Date', 'pet', 'ppt', 'aet', 'def', 'pdsi', 'q', 'soil', 'tmax', 'tmin', 'vpd']


In [32]:
missing = Terraclimate_training_df[climate_vars].isna().mean().sort_values(ascending=False)
print(missing.head(10))

pet     0.0
ppt     0.0
aet     0.0
def     0.0
pdsi    0.0
q       0.0
soil    0.0
tmax    0.0
tmin    0.0
vpd     0.0
dtype: float64


In [33]:
print(Terraclimate_training_df[climate_vars].describe().T[['min','max','mean']])

            min         max        mean
pet   52.700000  270.800020  175.166082
ppt    0.000000  374.200000   70.289301
aet    0.000000  173.400010   65.476028
def    0.000000  264.100000  109.690054
pdsi  -6.080000    6.290000   -1.367301
q      0.000000  129.000000    4.293261
soil   0.000000  162.200000    8.678979
tmax  14.589999   38.610000   30.221831
tmin  -3.300000   22.949999   16.313304
vpd    0.510000    3.310000    1.415572


### Extracting features for the validation dataset

In [12]:
Validation_df=pd.read_csv('./data/submission_template.csv')
Validation_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN


In [13]:
Validation_df.shape

(200, 6)

In [38]:
Validation_df = pd.read_csv('./data/submission_template.csv')  # Changed
Terraclimate_validation_df = Validation_df[['Latitude','Longitude','Sample Date']].copy()  # Changed

for var in climate_vars:
    tc_parameter = pd.read_csv(f"tc_{var}_bbox_2011_2015.csv")  # Changed
    temp_df = assign_nearest_climate(Validation_df, tc_parameter, var)  # Changed
    Terraclimate_validation_df[var] = temp_df[var]  # Changed

Terraclimate_validation_df.to_csv('./data/terraclimate_features_validation.csv', index=False)  # Changed

Mapping VPD values: 100%|██████████| 200/200 [00:00<00:00, 10340.86it/s]


In [14]:
# Preview File
Terraclimate_validation_df.head()

,Latitude,Longitude,Sample Date,pet
0,-32.043333,27.822778,01-09-2014,161.900009
1,-33.329167,26.077500,16-09-2015,177.600006
2,-32.991639,27.640028,07-05-2015,158.400009
3,-34.096389,24.439167,07-02-2012,130.000000
4,-32.000556,28.581667,01-10-2014,152.500000


In [14]:
Terraclimate_validation_df = pd.read_csv('./data/terraclimate_features_validation.csv')


In [15]:
# --- Date format diagnosis: put this RIGHT AFTER reading Water_Quality_df / Validation_df ---

import pandas as pd
import numpy as np

def diagnose_date_format(df, col="Sample Date", name="df"):
    s = df[col].astype(str).str.strip()

    # Basic stats
    print(f"\n=== Date Diagnosis: {name} ===")
    print("Total rows:", len(s))
    print("Unique examples:", s.dropna().head(10).tolist())

    # 1) Detect ISO-like yyyy-mm-dd / yyyy/mm/dd
    iso_mask = s.str.match(r"^\d{4}[-/]\d{1,2}[-/]\d{1,2}$", na=False)
    print("ISO-like (YYYY-MM-DD or YYYY/MM/DD) ratio:", float(iso_mask.mean()))

    # 2) Detect dash-based ambiguous dd-mm-yyyy or mm-dd-yyyy
    dmy_mask = s.str.match(r"^\d{1,2}-\d{1,2}-\d{4}$", na=False)
    print("Dash-based (??-??-YYYY) ratio:", float(dmy_mask.mean()))

    # 3) Unambiguous clues:
    # If first token > 12 -> must be day-first (DD-MM-YYYY)
    # If second token > 12 -> must be month-first (MM-DD-YYYY)
    dash = s[dmy_mask].str.split("-", expand=True)
    if len(dash) > 0:
        first = pd.to_numeric(dash[0], errors="coerce")
        second = pd.to_numeric(dash[1], errors="coerce")

        dd_first_clue = (first > 12).mean()   # implies DD-MM-YYYY
        mm_first_clue = (second > 12).mean()  # implies MM-DD-YYYY

        print("Clue: first token > 12 (=> DD-MM-YYYY) ratio:", float(dd_first_clue))
        print("Clue: second token > 12 (=> MM-DD-YYYY) ratio:", float(mm_first_clue))

    # 4) Parsing success comparison (only on non-null)
    nonnull = s.dropna()

    # Try parse as day-first and month-first (for ambiguous formats)
    parsed_dayfirst = pd.to_datetime(nonnull, dayfirst=True, errors="coerce")
    parsed_monthfirst = pd.to_datetime(nonnull, dayfirst=False, errors="coerce")

    ok_dayfirst = parsed_dayfirst.notna().mean()
    ok_monthfirst = parsed_monthfirst.notna().mean()

    print("Parse success (dayfirst=True) :", float(ok_dayfirst))
    print("Parse success (dayfirst=False):", float(ok_monthfirst))

    # 5) If both parse well, check how often they differ (ambiguous dates)
    both_ok = parsed_dayfirst.notna() & parsed_monthfirst.notna()
    if both_ok.any():
        diff_ratio = (parsed_dayfirst[both_ok] != parsed_monthfirst[both_ok]).mean()
        print("Ambiguity ratio (both parse but DIFFER):", float(diff_ratio))

        # Show a few ambiguous examples
        ambiguous_examples = nonnull[both_ok].loc[(parsed_dayfirst[both_ok] != parsed_monthfirst[both_ok])].head(10)
        if len(ambiguous_examples) > 0:
            print("Examples that flip depending on dayfirst:")
            for ex in ambiguous_examples.tolist():
                d1 = pd.to_datetime(ex, dayfirst=True, errors="coerce")
                d0 = pd.to_datetime(ex, dayfirst=False, errors="coerce")
                print(f"  {ex} -> dayfirst: {d1.date()} | monthfirst: {d0.date()}")

    # Recommended setting (simple heuristic)
    # If many first>12 => dayfirst True.
    # Else if many second>12 => dayfirst False.
    # Else fall back to higher parse success.
    rec = None
    if len(s[dmy_mask]) > 0:
        dash = s[dmy_mask].str.split("-", expand=True)
        first = pd.to_numeric(dash[0], errors="coerce")
        second = pd.to_numeric(dash[1], errors="coerce")
        if (first > 12).mean() > 0.001:
            rec = "dayfirst=True (likely DD-MM-YYYY)"
        elif (second > 12).mean() > 0.001:
            rec = "dayfirst=False (likely MM-DD-YYYY)"

    if rec is None:
        rec = "dayfirst=True" if ok_dayfirst >= ok_monthfirst else "dayfirst=False"

    print("RECOMMEND:", rec)


# Run diagnosis on both
diagnose_date_format(Water_Quality_df, col="Sample Date", name="Water_Quality_df")
diagnose_date_format(Validation_df, col="Sample Date", name="Validation_df")


=== Date Diagnosis: Water_Quality_df ===
Total rows: 9319
Unique examples: ['02-01-2011', '03-01-2011', '03-01-2011', '03-01-2011', '03-01-2011', '04-01-2011', '04-01-2011', '04-01-2011', '04-01-2011', '04-01-2011']
ISO-like (YYYY-MM-DD or YYYY/MM/DD) ratio: 0.0
Dash-based (??-??-YYYY) ratio: 1.0
Clue: first token > 12 (=> DD-MM-YYYY) ratio: 0.5877240047215366
Clue: second token > 12 (=> MM-DD-YYYY) ratio: 0.0
Parse success (dayfirst=True) : 1.0
Parse success (dayfirst=False): 0.41227599527846337
Ambiguity ratio (both parse but DIFFER): 0.9323269130661114
Examples that flip depending on dayfirst:
  02-01-2011 -> dayfirst: 2011-01-02 | monthfirst: 2011-02-01
  03-01-2011 -> dayfirst: 2011-01-03 | monthfirst: 2011-03-01
  03-01-2011 -> dayfirst: 2011-01-03 | monthfirst: 2011-03-01
  03-01-2011 -> dayfirst: 2011-01-03 | monthfirst: 2011-03-01
  03-01-2011 -> dayfirst: 2011-01-03 | monthfirst: 2011-03-01
  04-01-2011 -> dayfirst: 2011-01-04 | monthfirst: 2011-04-01
  04-01-2011 -> dayfirs